# 1 — ResNet-18 on CIFAR-10, in memory

The smallest useful case: a real pretrained model, a real dataset, and
activations small enough to keep in RAM.

`ActivationMapper.map()` with no path returns a `MemoryActivationStore`, which
holds one stacked tensor per layer. Use it when the whole extraction fits in
memory.

**Downloads on first run:** CIFAR-10 test split from the HuggingFace Hub
(~20 MB, parquet), ResNet-18 weights (~45 MB).

In [ ]:
import torch
import torchvision.transforms as T
from datasets import load_dataset
from torch.utils.data import Dataset
from torchvision.models import ResNet18_Weights, resnet18

from nnact import ActivationMapper, Sample

torch.manual_seed(0)

## The dataset

`nnact` reads any `Dataset` that yields `Sample(id=..., data=...)`. The `id` ties
a row of activations back to the input it came from, so use something stable —
here, the CIFAR-10 index plus its class name.

ResNet-18 expects 224×224 ImageNet-normalised input, so CIFAR's 32×32 images are
upscaled. That is the standard way to push CIFAR through an ImageNet backbone.

In [ ]:
transform = T.Compose(
    [
        T.Resize(224),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

cifar = load_dataset("uoft-cs/cifar10", split="test")
class_names = cifar.features["label"].names

print(f"CIFAR-10 test split: {len(cifar)} images")
print(f"columns: {cifar.column_names}")
print(f"classes: {class_names}")

In [ ]:
class CifarSamples(Dataset[Sample]):
    """Wraps the HuggingFace CIFAR-10 split so each item carries a stable id."""

    def __init__(self, base, n: int) -> None:
        self._base = base
        self._n = n

    def __len__(self) -> int:
        return self._n

    def __getitem__(self, idx: int) -> Sample:
        row = self._base[idx]
        image = transform(row["img"].convert("RGB"))
        return Sample(id=f"cifar_{idx:05d}_{class_names[row['label']]}", data=image)


n_images = 512
dataset = CifarSamples(cifar, n_images)

print(f"{len(dataset)} samples")
print(f"first id:    {dataset[0].id!r}")
print(f"input shape: {tuple(dataset[0].data.shape)}")

## The model

Any `nn.Module` works. `ActivationMapper.summary()` lists the layers you can
capture, so you can check what is hookable before extracting — pass `depth=2` to
descend into the blocks, or `available_layers()` for the bare list.

If you ask for a layer the model does not have, `map()` raises before the first
forward pass and suggests the nearby names.

In [ ]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)
mapper = ActivationMapper(model)

mapper.summary(depth=1)

## Extraction

Pick the layers you want and run. The mapper batches the data, switches the
model to eval mode, disables gradients, and shows a progress bar over the
batches.

It also records how the run was produced — model, layers, sample count, timing —
on `store.metadata`. For an HDF5 cache that travels with the file, so a store
reloaded months later can still say where it came from.

In [ ]:
LAYERS = ["layer3", "layer4", "avgpool", "fc"]

store = mapper.map(dataset, LAYERS, batch_size=64)

In [ ]:
store.metadata

## What came back

`store.summary()` reports the per-sample shape and the total bytes held for each
layer. `store.activations[name]` gives the whole stacked tensor.

Note how fast the early layers get expensive: `layer3` is two orders of
magnitude larger than `avgpool` for the same images.

In [ ]:
store.summary()

## Indexing by sample

A store is also a `Dataset`, so `store[i]` returns one sample's activations
across every layer. Order matches `layer_names`, and position `i` corresponds to
`sample_ids[i]`.

In [ ]:
sample = store[0]
print(f"store[0] is sample {store.sample_ids[0]!r}\n")
for activation in sample.activations:
    print(f"  {activation.layer_name:<10} {tuple(activation.tensor.shape)}")

## Cost of going earlier in the network

Layer choice dominates storage. Running the mapper over a single sample is
enough to read the per-sample shapes off `summary()`, which is all you need to
project the cost at any dataset size.

In [ ]:
probe_layers = ["layer1", "layer2", "layer3", "layer4", "avgpool", "fc"]
probe = mapper.map(
    CifarSamples(cifar, 1), probe_layers, batch_size=1, progress=False
).summary()

cost = probe[["shape", "elements"]].copy()
cost["KB / sample"] = probe["elements"] * 4 / 1024
cost["MB / 512 imgs"] = probe["elements"] * 4 * 512 / 1024**2
cost["GB / 10k imgs"] = probe["elements"] * 4 * 10_000 / 1024**3
cost.round(2)